# Notebook 04 (R) — Build the DM Domain

**Module:** Building SDTM Domains I · **Pairs with:** `04_build_dm_domain_SAS.sas`

Turn `dm_raw.csv` (+ `ex_raw.csv`) into a compliant SDTM **DM** dataset.
DM is a **Special Purpose** domain: **one row per subject**, and it has no `--SEQ`.

You will:
1. build `USUBJID`
2. derive `AGE` from the birth date
3. apply Controlled Terminology to `SEX`, `RACE`, `ETHNIC`
4. derive `ARMCD` from `ARM`
5. bring `RFSTDTC` / `RFENDTC` across from EX
6. assemble the variables in SDTM order and check your work

**Spec:** `../../data/mapping_specification.md` (§2) · **Target:** `../../data/sdtm/dm.csv`

> Attempt it first, then compare against the target at the end.

## 0. Setup and read the raw data

`SITEID` and `SUBJID` as character (Notebook 03).

**The raw dates are `DD-MMM-YYYY`** (e.g. `14-MAY-1969`) — the format the CRF collects.
They are NOT ISO 8601: converting them is one of the mapping jobs. readr leaves them
as character because it cannot guess that format, which is exactly what we want.

In [ ]:
library(readr)
library(dplyr)

datapath <- "/Volumes/D Drive/SDTM Training/Bootcamp/data"   # <-- EDIT if needed

dm_raw <- read_csv(file.path(datapath, "dm_raw.csv"),
  col_types = cols(SITEID = col_character(), SUBJID = col_character(), .default = col_guess()))

ex_raw <- read_csv(file.path(datapath, "ex_raw.csv"),
  col_types = cols(SITEID = col_character(), SUBJID = col_character(), .default = col_guess()))

# end-of-study comes from the DISPOSITION form, not demographics
ds_raw <- read_csv(file.path(datapath, "ds_raw.csv"),
  col_types = cols(SITEID = col_character(), SUBJID = col_character(), .default = col_guess()))

glimpse(dm_raw)

## 1. Reference dates from EX

`RFSTDTC` = date of **first dose** — this defines Study Day 1 everywhere in the study.
`RFENDTC` = date of **last dose**.

Our EX has exactly one row per subject, so min/max is trivial — but written this way
it still works if a subject has several dosing records.

In [ ]:
ref_dates <- ex_raw |>
  group_by(SITEID, SUBJID) |>
  # parse FIRST — DD-MMM-YYYY text does not sort chronologically!
  summarise(RFSTDTC = min(crf_date(EXSTDTC)), RFENDTC = max(crf_date(EXENDTC)), .groups = "drop")

ref_dates

## 2. Parsing the raw dates

`as.Date()` cannot read `14-MAY-1969` on its own — you must tell it the format.
`%d` = day, `%b` = abbreviated month name, `%Y` = 4-digit year.

In [ ]:
crf_date <- function(x) as.Date(x, format = "%d-%b-%Y")

# check it round-trips to ISO
crf_date("14-MAY-1969")            # 1969-05-14
format(crf_date("14-MAY-1969"), "%Y-%m-%d")

## 3. A helper for AGE

`AGE` is **completed years** from birth date to a reference date (here, informed
consent). The subtraction at the end handles the case where the birthday has not
yet occurred in the reference year.

Watch the edge case: born 1969-05-14, consented 2024-02-20 → **54**, not 55.

In [ ]:
age_years <- function(birth, ref) {
  b <- crf_date(birth); r <- crf_date(ref)   # raw dates are DD-MMM-YYYY
  years <- as.integer(format(r, "%Y")) - as.integer(format(b, "%Y"))
  # not had their birthday yet this year? subtract one
  before_bday <- (format(r, "%m%d") < format(b, "%m%d"))
  years - as.integer(before_bday)
}

# quick sanity check
age_years("14-MAY-1969", "20-FEB-2024")   # 54

## 4. Build DM

One `mutate()` does the whole mapping. Note:
- `paste(..., sep = "-")` builds `USUBJID` — unique across the whole study
- `case_when()` applies Controlled Terminology (with an explicit fallback)
- `toupper(trimws())` normalises the free-text `RACE` / `ETHNIC`
- `format(x, "%Y-%m-%d")` writes dates back out as **ISO character**

In [ ]:
dm <- dm_raw |>
  left_join(ref_dates, by = c("SITEID", "SUBJID")) |>
  left_join(ds_raw |> select(SITEID, SUBJID, EOSDT), by = c("SITEID", "SUBJID")) |>
  mutate(
    DOMAIN   = "DM",
    USUBJID  = paste(STUDYID, SITEID, SUBJID, sep = "-"),

    # --- age FIRST: mutate() runs top-to-bottom, and the lines below
    #     overwrite RFICDTC with its ISO form. Derive AGE while the raw
    #     DD-MMM-YYYY value is still there.
    AGE  = age_years(BRTHDTC, RFICDTC),
    AGEU = "YEARS",

    # --- reference dates (SDTM wants character ISO 8601) ---
    RFSTDTC  = format(RFSTDTC,  "%Y-%m-%d"),
    RFENDTC  = format(RFENDTC,  "%Y-%m-%d"),
    RFXSTDTC = RFSTDTC,                          # first study-treatment exposure
    RFXENDTC = RFENDTC,                          # last  study-treatment exposure
    RFICDTC  = format(crf_date(RFICDTC), "%Y-%m-%d"),
    RFPENDTC = format(crf_date(EOSDT),   "%Y-%m-%d"),   # end of participation (from ds_raw)
    DTHDTC   = NA_character_,                    # no deaths in this study
    DTHFL    = NA_character_,

    # --- controlled terminology ---
    SEX = case_when(SEX == 1 ~ "M",
                    SEX == 2 ~ "F",
                    TRUE     ~ NA_character_),
    RACE   = toupper(trimws(RACE)),
    ETHNIC = toupper(trimws(ETHNIC)),

    # --- arm ---
    ARMCD    = case_when(ARM == "Drug A"  ~ "A",
                         ARM == "Placebo" ~ "P",
                         TRUE             ~ NA_character_),
    ACTARM   = ARM,        # actual = planned here: everyone got their assignment
    ACTARMCD = ARMCD
  ) |>
  # --- SDTM variable order ---
  select(STUDYID, DOMAIN, USUBJID, SUBJID,
         RFSTDTC, RFENDTC, RFXSTDTC, RFXENDTC, RFICDTC, RFPENDTC,
         DTHDTC, DTHFL, SITEID, AGE, AGEU, SEX, RACE, ETHNIC,
         ARMCD, ARM, ACTARMCD, ACTARM, COUNTRY) |>
  arrange(USUBJID)

dm

> **Trap — `mutate()` is sequential.** The lines run top to bottom, so once
> `RFICDTC` has been overwritten with its ISO form, `crf_date()` can no longer parse
> it and `AGE` comes out as `NA`. Derive anything that depends on a raw value
> **before** you overwrite that value. Order matters.

## 5. Check your work

The same checks listed in `mapping_specification.md` §9.

In [ ]:
# Check 1 — one row per subject, USUBJID unique
cat("rows:", nrow(dm), "| distinct USUBJID:", n_distinct(dm$USUBJID), "\n")

# Check 2 — no missing Required variables
dm |> summarise(across(c(USUBJID, SEX, ARMCD, ARM, COUNTRY, SITEID), ~ sum(is.na(.x))))

In [ ]:
# Check 3 — controlled values only
dm |> count(SEX)
dm |> count(RACE)
dm |> count(ARMCD, ARM)

## 6. Compare against the reference

`../../data/sdtm/dm.csv` is the finished answer. Any difference is either a bug in
your mapping — or a deliberate choice you should be able to defend.

In [ ]:
target <- read_csv(file.path(datapath, "sdtm", "dm.csv"),
  col_types = cols(.default = col_character(), AGE = col_integer()))

mine <- dm |> mutate(across(everything(), as.character)) |>
              mutate(AGE = as.integer(AGE))

cat("same columns: ", identical(names(mine), names(target)), "\n")
cat("same values:  ", isTRUE(all.equal(as.data.frame(mine), as.data.frame(target))), "\n")

## YOUR TURN — exercises

Solutions: `../../answer-keys/04_build_dm_answers.md`

**Exercise 1.** `AGE` is derived at informed consent. Some studies derive it at first
dose instead. Create `AGE_ALT` using `RFSTDTC` and compare — which subjects get a
different age, and why?

**Exercise 2.** Our code trusts that `toupper(trimws(RACE))` always lands on a valid CT
value. Write a check that lists any `RACE` not in the allowed set:
`WHITE`, `ASIAN`, `BLACK OR AFRICAN AMERICAN`, `AMERICAN INDIAN OR ALASKA NATIVE`,
`NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER`, `OTHER`, `UNKNOWN`, `NOT REPORTED`.
It should return zero rows.

**Exercise 3.** Subject 01/004 discontinued early. Using `EOSSTAT` and `EOSDT` from
`ds_raw`, list each subject with their completion status and days on treatment
(`RFENDTC − RFSTDTC + 1`).

**Exercise 4 (stretch).** Which subject is youngest, and which oldest? Does age differ
meaningfully between the two arms? (A real study would check this for balance.)

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4 (stretch)